# homography: ArUco corners -> top-down warp + 8x8 grid (M1)

The board has four ArUco markers (DICT_4X4_50, ids 0..3 next to a1, h1, h8, a8) printed in
the frame around the playing area. Their positions on the board are known from
`gambit_bringup/config/board.yaml`, so detecting them in the camera image gives four
point correspondences -> a homography from the image to a canonical top-down view
("warp"). In the warp every square is an axis-aligned `square_px` x `square_px` cell, so
the 8x8 grid and the square crops (`capture.ipynb`) are pure arithmetic.

Coordinate frames used here:
* **board** (meters): origin at the outer corner of a1 of the playing area, `u` along the
  files (a -> h), `v` along the ranks (1 -> 8). The playing area is `[0, 8s] x [0, 8s]`.
* **warp** (pixels): the top-down image, `8 * square_px + 2 * margin_px` square. With
  `view: black_bottom` (default, camera on the black side) rank 8 is at the bottom and the
  a-file on the right, like the camera sees it.
* **image** (pixels): the raw (ideally undistorted) camera frame.

Milestone: M1 (calibration + grid overlay). Deliverable: grid overlay on the live image and
the square-center pixel error.

**TODO**
- [ ] board geometry loading (`load_board_geometry`, `marker_centers_board`)
- [ ] board <-> warp affine map and the 8x8 grid helpers
- [ ] marker detection, marker centers, `compute_homography`, `warp_board`
- [ ] grid model in the image, overlay drawing, square-center error metric
- [ ] `render_synthetic_board` (lets you test everything without a camera)
- [ ] fallback when only 3 markers are visible (a hand covers one): affine from 3 points,
      or reuse the last good H (the camera is fixed)
- [ ] use all 16 marker corners instead of 4 centers once the printed marker orientation
      on the real board is confirmed (more points -> least-squares H, less jitter)
- [ ] undistort with the M1 intrinsics before detection (`calibration.ipynb: undistort`)
- [ ] measure the square-center error on the real board (`run_grid_overlay.ipynb`)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import chess
import numpy as np
import yaml

%run ./config.ipynb

## Board geometry
Read from `board.yaml` (owned by `gambit_bringup`; on the real board measure with a ruler
and update it there). Marker centers sit `border / 2` outside the nearest playing-area
corner on both axes.

In [ ]:
MARKER_CORNERS = ("a1", "h1", "h8", "a8")


@dataclass
class BoardGeometry:
    square: float        # square edge (m)
    border: float        # frame width around the playing area (m)
    marker_size: float   # marker edge (m)
    dictionary: str      # e.g. "DICT_4X4_50"
    ids: dict            # corner name -> marker id


def load_camera_cfg(path: str | Path = "configs/camera.yaml"):
    """Thin wrapper around load_config for camera.yaml."""
    # TODO: implement (one line)
    raise NotImplementedError


def load_board_geometry(board_yaml: str | Path | None = None) -> BoardGeometry:
    """Parse gambit_bringup/config/board.yaml (path relative to gambit_vision/).

    None -> the path in camera.yaml: board_yaml.
    """
    # Design:
    # - yaml.safe_load the file (resolve with gv_path); pick board.square, board.border,
    #   markers.size, markers.dictionary, markers.ids into a BoardGeometry.
    # - Keep floats as floats; ids stays a {corner name: id} dict.
    # TODO: implement
    raise NotImplementedError


def marker_centers_board(geom: BoardGeometry) -> dict[str, np.ndarray]:
    """Marker center (u, v) in board meters for each corner name in MARKER_CORNERS."""
    # Design: playing area is [0, 8s]^2; each marker center is border/2 outside the nearest
    # corner on both axes (e.g. a1 -> (-border/2, -border/2)).
    # TODO: implement
    raise NotImplementedError

## Board meters <-> warp pixels
An affine map (scale `k = square_px / square`, a flip depending on `view`, an offset of
`margin_px`). Written as a 3x3 matrix so it composes with homographies.

In [ ]:
def warp_size(warp) -> int:
    """Side length (px) of the square top-down image: 8 * square_px + 2 * margin_px."""
    # TODO: implement
    raise NotImplementedError


def board_to_warp_matrix(geom: BoardGeometry, warp) -> np.ndarray:
    """3x3 matrix mapping board meters (u, v, 1) to warp pixels (x, y, 1)."""
    # Design:
    # - Scale k = square_px / square, offset margin_px, playing area 8 * square_px wide.
    # - view "white_bottom": a-file left, rank 1 at the bottom -> x grows with u, y shrinks with v.
    # - view "black_bottom" (default, camera on the black side): a-file right, rank 8 at the
    #   bottom -> x shrinks with u, y grows with v.
    # - Unknown view -> ValueError. Work out the offsets so the playing area lands at
    #   [margin, margin + 8*square_px] on both axes.
    # TODO: implement
    raise NotImplementedError


def _apply(Hm: np.ndarray, pts) -> np.ndarray:
    """Apply a 3x3 projective matrix to points of shape (..., 2); returns the same shape."""
    # Design: homogeneous coordinates (append 1, multiply, divide by the last component).
    # cv2.perspectiveTransform does the same but wants (N, 1, 2) float arrays.
    # TODO: implement
    raise NotImplementedError


def board_to_warp(uv, geom: BoardGeometry, warp) -> np.ndarray:
    """Board meters -> warp pixels (any (..., 2) shape)."""
    # TODO: implement (compose _apply with board_to_warp_matrix)
    raise NotImplementedError


def warp_to_board(xy, geom: BoardGeometry, warp) -> np.ndarray:
    """Warp pixels -> board meters (inverse matrix)."""
    # TODO: implement
    raise NotImplementedError

## The 8x8 grid in warp coordinates

In [ ]:
def square_center_board(sq: int, geom: BoardGeometry) -> np.ndarray:
    """(u, v) meters of the center of square sq: (file + 0.5, rank + 0.5) * square."""
    # TODO: implement (chess.square_file / chess.square_rank)
    raise NotImplementedError


def square_cell_warp(sq: int, geom: BoardGeometry, warp) -> tuple[int, int, int, int]:
    """Axis-aligned cell (x0, y0, x1, y1) of square `sq` in the warp image (x1, y1 exclusive)."""
    # Design: map the two opposite board corners of the square to warp pixels, take
    # min/max per axis (the flip depends on view), round to int.
    # TODO: implement
    raise NotImplementedError


def square_centers_warp(geom: BoardGeometry, warp) -> np.ndarray:
    """(64, 2) square centers in warp pixels, square-index order."""
    # TODO: implement
    raise NotImplementedError


def grid_points_warp(geom: BoardGeometry, warp) -> np.ndarray:
    """(9, 9, 2) grid line intersections in warp pixels, indexed [rank_line, file_line]."""
    # Design: meshgrid of 0..8 line indices times square, then board_to_warp.
    # TODO: implement
    raise NotImplementedError

## Marker detection and the homography
`cv2.aruco.ArucoDetector` returns 4 corners per marker. The marker center is taken as the
intersection of the two diagonals, which (unlike the mean of the corners) is exactly the
projected center under perspective.

In [ ]:
def get_dictionary(name: str):
    """cv2.aruco predefined dictionary by name, e.g. "DICT_4X4_50"."""
    # TODO: implement (one line)
    raise NotImplementedError


def detect_markers(image: np.ndarray, dictionary: str = "DICT_4X4_50") -> dict[int, np.ndarray]:
    """Detect ArUco markers; returns {id: (4, 2) corners in image pixels} ({} if none)."""
    # Design:
    # - Grayscale, then cv2.aruco.ArucoDetector(dictionary, DetectorParameters()).detectMarkers.
    # - Output ids can be None; corners come as (1, 4, 2) per marker.
    # Decision for you: tune DetectorParameters (e.g. corner refinement) if detections jitter.
    # TODO: implement
    raise NotImplementedError


def marker_center(corners: np.ndarray) -> np.ndarray:
    """Intersection of the diagonals corner0-corner2 and corner1-corner3."""
    # Design: write both diagonals as p0 + t*(p2-p0) and p1 + s*(p3-p1), solve the 2x2
    # linear system for t (np.linalg.solve). Why not the mean: perspective shifts the mean
    # away from the true projected center, the diagonal intersection is invariant.
    # TODO: implement
    raise NotImplementedError


def compute_homography(image: np.ndarray, geom: BoardGeometry, warp,
                       markers: dict[int, np.ndarray] | None = None):
    """Homography H mapping image pixels -> warp pixels, from the 4 corner markers.

    Returns (H, info) where info has the detected ids and the image-space marker centers
    ({"ids": sorted ids, "centers_image": (4, 2)}).
    Raises RuntimeError if a corner marker is missing (name the missing corners).
    """
    # Design:
    # - markers=None -> detect_markers(image, geom.dictionary).
    # - For each corner in MARKER_CORNERS: src = marker_center in the image,
    #   dst = marker_centers_board -> board_to_warp.
    # - cv2.findHomography(src, dst) with 4 exact points (method 0); RANSAC only pays off
    #   once you use more points (all 16 marker corners).
    # Decisions for you: 3-marker fallback (cv2.getAffineTransform or reuse the last good H).
    # TODO: implement
    raise NotImplementedError


def warp_board(image: np.ndarray, H: np.ndarray, warp) -> np.ndarray:
    """Top-down image of size warp_size(warp) x warp_size(warp)."""
    # Design: cv2.warpPerspective with linear interpolation.
    # TODO: implement
    raise NotImplementedError

## Grid model in the camera image, overlay, error metric
The inverse homography maps warp points back into the image. The M1 metric is the pixel
distance between predicted square centers and ground-truth centers (clicked by hand on the
real board, or known exactly in simulation / synthetic images).

In [ ]:
def to_image(points_warp, H: np.ndarray) -> np.ndarray:
    """Map warp-pixel points into image pixels with inv(H)."""
    # TODO: implement
    raise NotImplementedError


def grid_model_image(H: np.ndarray, geom: BoardGeometry, warp) -> dict[str, np.ndarray]:
    """{"centers": (64, 2), "corners": (9, 9, 2)} in image pixels."""
    # TODO: implement
    raise NotImplementedError


def draw_grid_overlay(image: np.ndarray, H: np.ndarray, geom: BoardGeometry, warp,
                      labels: list[str] | None = None, color=(0, 255, 0),
                      names: bool = True) -> np.ndarray:
    """Copy of `image` with the 8x8 grid, and square names (or 64 labels) drawn on it."""
    # Design:
    # - Draw the 9 rank lines and 9 file lines through the projected grid corners
    #   (cv2.polylines, anti-aliased), not straight lines between the outer corners:
    #   if lens distortion is left in, the mismatch becomes visible.
    # - Text at each projected center (cv2.putText): label if given (skip "empty"),
    #   else the square name.
    # TODO: implement
    raise NotImplementedError


def square_center_pixel_error(pred: np.ndarray, true: np.ndarray) -> dict[str, float]:
    """Euclidean error between predicted and true square centers (N, 2): mean, rms, max (px)."""
    # TODO: implement
    raise NotImplementedError

## Synthetic board (for the self-check and for debugging without a camera)
Draws the board in warp coordinates: frame, light/dark squares and the four markers, each
with a one-module white quiet zone (ArUco detection needs it; print them that way too).

In [ ]:
def render_synthetic_board(geom: BoardGeometry, warp, light=(190, 220, 235), dark=(60, 85, 120),
                           frame=(35, 50, 70)) -> np.ndarray:
    """BGR top-down board image in warp coordinates (size warp_size(warp))."""
    # Design:
    # - White canvas; fill the frame rectangle (playing area + border) with `frame` color.
    # - Fill each square cell (square_cell_warp); a1 is dark, i.e. (file + rank) odd is light.
    # - Paste each marker (cv2.aruco.generateImageMarker, side = marker_size * k px) at its
    #   warp center, with a white quiet zone of one module (side / 6 for a 4x4 marker).
    # - Raise ValueError if a marker would fall outside the image (margin too small).
    # TODO: implement
    raise NotImplementedError

## Self-check
Render a synthetic board, push it through a known perspective (the "camera"), recover the
homography from the detected markers and compare predicted square centers with the truth.

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - load_board_geometry(): ids == {"a1": 0, "h1": 1, "h8": 2, "a8": 3}, square == 0.025;
#   camera.yaml markers.ids agrees with board.yaml.
# - Frame conventions with warp = SimpleNamespace(square_px=60, margin_px=90, view="black_bottom"):
#   a1 center at (90 + 7.5*60, 90 + 0.5*60), h8 center at (90 + 0.5*60, 90 + 7.5*60);
#   square_cell_warp(a1) == (510, 90, 570, 150); white_bottom puts a1 center at (120, 540).
# - warp_to_board(board_to_warp(p)) == p.
# - marker_center of the unit square [[0,0],[2,0],[2,2],[0,2]] is (1, 1).
# - Synthetic camera: render_synthetic_board, warp it into a 640x480 "camera" image with a
#   known cv2.getPerspectiveTransform (corners moved inward by different amounts), then
#   compute_homography finds ids [0, 1, 2, 3] and square_center_pixel_error against the
#   true projected centers has max < 1.5 px.
# - draw_grid_overlay changes the image but keeps its shape; warp_board gives (n, n, 3).
# - Covering the a1 marker with grey -> compute_homography raises RuntimeError mentioning "a1".
print("homography: self-check not written yet (see plan above)")